# 08 — App Gradio do assistente clínico

Orquestrador final: carrega modelo fine-tuned + Chroma + DB mock e expõe a UI Gradio via ngrok/share.

## Pré-requisitos

1. ✅ `05_gerar_dados_mock.ipynb` rodado → `hospital.db` existe.
2. ✅ `06_indexar_protocolos.ipynb` rodado → Chroma existe.
3. ✅ `03_treinar_qlora.ipynb` rodado → adapter LoRA salvo (opcional — sem adapter usa modelo base).
4. Pasta `lib/` em `/MyDrive/AssistenteHospitalar/lib/`.
5. Colab com GPU (A100/L4 recomendado).

**Importante:** rode direto no navegador Colab, não via VSCode tunnel (sessões longas com Gradio são frágeis no tunnel).

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes \
    langchain langchain-community langchain-huggingface langgraph \
    chromadb sentence-transformers gradio python-dotenv

In [ ]:
import os, sys
from pathlib import Path
from google.colab import drive
from dotenv import load_dotenv
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/AssistenteHospitalar'
LIB_PATH   = DRIVE_BASE           # lib/ deve estar em /MyDrive/AssistenteHospitalar/lib/
DB_PATH    = f'{DRIVE_BASE}/files/hospital.db'
CHROMA_DIR = f'{DRIVE_BASE}/files/chroma'
COLLECTION = 'protocolos_saude_mulher'

sys.path.insert(0, LIB_PATH)
os.environ['HOSPITAL_DB_PATH'] = DB_PATH
os.environ['DRIVE_BASE']       = DRIVE_BASE

load_dotenv(f'{DRIVE_BASE}/.env')
login(token=os.getenv('HF_TOKEN'))
print('Setup ok.')

In [ ]:
# Carrega LLM (base + adapter mais recente, se houver)
# Defaults atualizados (max_new_tokens=256, repetition_penalty=1.2, no_repeat_ngram_size=4)
# mitigam loops degenerativos observados na avaliação da run 0217
from lib.llm import load_finetuned, build_chat_model

model, tokenizer = load_finetuned()        # auto-descobre adapter em files/finetune/
chat_model = build_chat_model(model, tokenizer)   # usa defaults novos
print('LLM pronto.')

In [ ]:
# Carrega Chroma (vector store dos protocolos)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
)
vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print(f'Chroma: {vectorstore._collection.count()} chunks.')

In [ ]:
# Conecta no DB mock + monta agente + compila os 4 workflows LangGraph
from lib import db, tools
from lib.agent import build_agent
from lib.workflows import (
    build_triagem_workflow, build_violencia_workflow,
    build_obstetrico_workflow, build_prevencao_workflow,
)

conn = db.connect()
tools_list = tools.build_langchain_tools(conn, retriever)
agent = build_agent(chat_model, tools_list)

workflows = {
    'triagem':    build_triagem_workflow(chat_model, conn, retriever),
    'violencia':  build_violencia_workflow(chat_model, conn, retriever),
    'obstetrico': build_obstetrico_workflow(chat_model, conn, retriever),
    'prevencao':  build_prevencao_workflow(chat_model, conn, retriever),
}

print(f'Agente pronto com {len(tools_list)} ferramentas:')
for t in tools_list:
    print(f'  - {t.name}')
print(f'\nWorkflows LangGraph compilados: {list(workflows.keys())}')

In [ ]:
# Smoke test antes do UI — confirma que o agente responde
from lib.agent import run_consulta

out = run_consulta(
    agent,
    pergunta='Quais critérios para repetir citologia em paciente <25a com LSIL?',
)
print('RESPOSTA:\n', out['resposta'])
print('\nTOOL CALLS:')
for c in out['tool_calls']:
    print(f'  - {c["tool"]}({c["args"]}) ')

In [ ]:
# Lança o UI Gradio com os 4 workflows habilitados (share=True abre ngrok)
from lib.ui import build_ui

app = build_ui(
    agent,
    conn,
    default_usuario='dr_residente_demo',
    workflows=workflows,            # 4 tabs novas: triagem, violência, obstétrico, prevenção
)
app.launch(share=True, debug=False, show_error=True)

## Cenários para demonstrar no vídeo

A UI agora tem **5 tabs**:

### 💬 Consulta livre
Chat com o agente LangChain. Ele decide quais ferramentas chamar.
- _"Posologia de ácido fólico para prevenção de defeito do tubo neural"_
- _"Quais critérios para repetir citologia em paciente <25a com LSIL?"_

### 🩺 Triagem Ginecológica (workflow LangGraph)
Cole queixa no textarea → fluxo executa parse → análise risco → urgência → exames → agendamento.
- **Emergência:** _"Paciente 32a, sangramento intenso há 3 dias, dor pélvica forte irradiando para ombro, atraso menstrual de 8 semanas."_
- **Rotina:** _"Paciente 28a, corrimento amarelado com odor há 5 dias, sem febre, sem dor pélvica."_

### 🛡️ Detecção de Violência (2 sub-tabs)
- **Workflow LangGraph completo:** descrição livre → matriz de pontuação → protocolo de segurança → SINAN
- **Checklist heurístico:** marca sinais clínicos → score determinístico (sem LLM)

Exemplo workflow: _"Paciente 28a comparece com lesões equimóticas em locais não-expostos, em múltiplas fases de cicatrização. Acompanhante recusou deixar a paciente sozinha. Histórico de 3 atendimentos prévios. Isolamento social progressivo."_ + selecione paciente na sidebar + marque `Confirmação clínica`.

### 🤰 Atendimento Obstétrico (workflow LangGraph)
Descrição clínica + IG (opcional, extraído da descrição se vazio).
- **Pré-eclâmpsia:** _"Gestante 34a, G3P2A0, IG 32 semanas pela DUM. Cefaleia intensa há 24h, escotomas, edema súbito de face, dor epigástrica em barra. HAS gestacional na semana 28."_
- **1ª consulta:** _"Primigesta 24a, IG 12 semanas pela DUM, sem antecedentes mórbidos."_

### 📅 Prevenção e Rastreamento (workflow LangGraph)
Selecione paciente na sidebar (ideal: alguma com mamografia ou papanicolau em atraso, que o painel da sidebar destaca em 🔴) → clique "Gerar plano preventivo".

### LGPD em ação
Selecione uma paciente com registro prévio de violência (sidebar mostra "🔒 N registros") → no chat livre, pergunte sobre o histórico — o agente exige motivo clínico via `consultar_violencia`. O acesso fica em `log_acesso`.

## Encerrando

Antes de fechar o notebook:
```python
app.close()
conn.close()
```

In [ ]:
import shutil, os, json, re
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

# Apaga Chroma vazio
if os.path.exists('/content/chroma_local'):
    shutil.rmtree('/content/chroma_local')

# Carrega JSON de fontes (39 PDFs estruturados)
with open(f'{DRIVE_BASE}/files/fontes_saude_mulher_v2.json') as f:
    docs = json.load(f)

def chunk_text(text, size=6000, overlap=400):
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    if len(text) <= size: return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            cut = text.rfind('\n\n', start, end)
            if cut > start + size // 2: end = cut
        chunks.append(text[start:end].strip())
        if end >= len(text): break
        start = end - overlap
    return [c for c in chunks if len(c) > 200]

lc_docs = []
for d in docs:
    for i, ch in enumerate(chunk_text(d['content'])):
        lc_docs.append(Document(
            page_content=ch,
            metadata={
                'doc_id': d['filename'],
                'chunk_id': f"{d['filename']}::{i}",
                'category': d['category'],
                'sensitive': d['sensitive'],
                'name': d['name'],
            },
        ))

print(f'Indexando {len(lc_docs)} chunks em /content/chroma_local ...')
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name=COLLECTION,
    persist_directory='/content/chroma_local',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print(f'✅ Chroma local: {vectorstore._collection.count()} chunks.')


: 